# SparkCaster Time Series Forecasting System

SparkCaster gateway: `http://spark-gateway.kubeflow.svc.cluster.local:8888`

## Overview
A **distributed time series forecasting system on SparkCaster**. Each
(metric × grouping × group) series is one task; models are fit on Spark
executors and the best model per series is selected automatically.

- **Cluster:** ~20 executors × 4 cores (dynamic allocation 5–25), 16 GB/executor
- **Data source:** Nessie catalog table `sandbox.sandbox_finance.dcgm_metrics_summary_imputed`
- **Data stays in Spark** until results are collected (no early `.toPandas()`)
- **Python 3.11**

## Bootstrap (driver vs executor are intentionally separate)
A single dependency manifest `PACKAGES = {"driver": [...], "executor": [...]}`
drives three distinct install paths:
1. **Driver setup** (`bootstrap_driver`) — interactive kernel, has internet, `pip install --user`; full analysis/vis stack.
2. **Executor prep** — a **wheelhouse** zip of the minimal executor libs, built on the driver and shipped via `SparkFiles`.
3. **Executor runtime** — each task installs from that wheelhouse **offline** (workers have no internet).

## Models (5, fit per series on executors)
1. **Exponential Smoothing** (additive trend + seasonal)
2. **ARIMA** `(5,1,0)`
3. **SARIMA** `(1,1,1)(1,1,1,7)` (weekly seasonality)
4. **Prophet** (native 80% intervals)
5. **Holt-Winters** (damped, multiplicative seasonal)

Best model chosen by lowest MAE. **P10/P50/P90 prediction intervals** and
**`[0,1]` bounds for utilization metrics** are computed on the executors, so
the distributed output matches what plotting/export consume (one canonical
result schema, documented in `forecast_time_series_row`).

- **Train/test split:** `TRAIN_SPLIT = 0.7`  ·  **Horizon:** `FORECAST_DAYS = 1100` (~3 years)
- These are the single source of truth; `CONFIG` mirrors them.

## Metrics forecasted (8)
`gpu_util_p50`, `tensor_util_p50/p95/p99`, `chip_power_p50/p95`,
`redfish_power_p50/p95`.
*(tflops_* metrics are excluded — not present in this table.)*

## Grouping strategies (8)
`All`, `product_resolved`, `product_segment`, `customer_segment`,
`region_summary`, `region_summary+product_segment`,
`region_summary+product_resolved`, `product_segment+product_resolved`.

## Outputs → CoreWeave Object Storage (LOTA)
Final pandas frames are exported as **CSV + XLSX** to
`s3://jbok-sandbox-test/jbok/time-series-fcst-sparkcaster/<timestamp>/`
(one run-scoped timestamp), then downloaded from Cloud Console. Datasets:

| Object (`.csv` and `.xlsx`) | Contents |
|---|---|
| `all_models_results` | Metrics for every model × series |
| `best_models_results` | Best model per series |
| `forecast_daily_values` | Daily P10/P50/P90 forecast (best model) |
| `forecast_monthly_values` | Monthly aggregate (best model) |
| `forecast_monthly_values_all` | Monthly aggregate (all models) |
| `forecast_monthly_values_all_with_history` | Monthly actuals + forecast (all models) |

## How to run
**Restart & Run All**, top to bottom. You will be prompted (via `getpass`) for:
- the encrypted-keyring master password (once), then CAIOS credentials (for Nessie);
- your CoreWeave Object Storage **access key / secret** at the export helper
  (or set `CW_S3_ACCESS_KEY` / `CW_S3_SECRET_KEY` in the kernel to skip the prompt).

## Architecture
1. `create_time_series_tasks` builds one Spark row per (metric, grouping, group_key) with its `(day, value)` array.
2. `forecast_time_series_row` runs on each executor (offline dependency install → fit 5 models → intervals + bounds → canonical JSON).
3. `run_sparkcaster_forecasting` distributes via RDD `map` and collects results to the driver.
4. Results are expanded to pandas, aggregated to daily/monthly grains, and exported to object storage.


In [13]:
# ── SETUP: logging helpers + dependency manifest + DRIVER bootstrap ──────────
import importlib, subprocess, sys, os, site

# --- Logging helpers (used across the notebook instead of ad-hoc print blocks) ---
def log(*args, **kwargs):
    """Drop-in for print(); central hook for future structured logging."""
    print(*args, **kwargs)

def log_section(title, char="=", width=80):
    """Banner header: rule / title / rule."""
    print(char * width)
    print(title)
    print(char * width)

# --- Single dependency manifest, shared by every install path ----------------
# WHY driver vs executor differ:
#   * DRIVER (this kernel): interactive, has internet -> pip install --user.
#     Installs the full analysis/vis stack used only on the driver.
#   * EXECUTOR (Spark workers): no internet -> installed OFFLINE from a
#     wheelhouse zip (see the wheelhouse cell) at task runtime (see the
#     forecasting cell). Only the minimal libs each task needs to fit models.
PACKAGES = {
    "driver": [
        ("keyring", "keyring"),
        ("ipython-secrets", "ipython_secrets"),
        ("oauth2client", "oauth2client"),
        ("pyarrow", "pyarrow"),
        ("fsspec", "fsspec"),
        ("s3fs", "s3fs"),
        ("scipy", "scipy"),
        ("statsmodels", "statsmodels"),
        ("matplotlib", "matplotlib"),
        ("scikit-learn", "sklearn"),
        ("keyrings.cryptfile", "keyrings"),   # plugin under the 'keyrings' pkg
        ("bokeh==3.6.2", "bokeh"),
        ("jupyter_bokeh", "jupyter_bokeh"),
        ("panel==1.5.2", "panel"),
        ("holoviews==1.19.0", "holoviews"),
        ("hvplot==0.10.0", "hvplot"),
        ("datashader==0.16.3", "datashader"),
        ("dask[dataframe]==2024.9.1", "dask"),
        ("distributed==2024.9.1", "distributed"),
        ("reportlab", "reportlab"),
        ("prophet", "prophet"),
        ("openpyxl", "openpyxl"),
        ("tqdm", "tqdm"),
    ],
    # Minimal libs each executor needs to fit models; downloaded to the
    # wheelhouse on the driver and pip-installed offline on the workers.
    "executor": ["statsmodels", "scipy", "pandas", "numpy", "patsy"],
}

def ensure_user_site():
    """Make user site-packages / user bin visible in this kernel."""
    user_site = site.getusersitepackages()
    if user_site and user_site not in sys.path:
        sys.path.insert(0, user_site)
    user_bin = os.path.expanduser("~/.local/bin")
    if user_bin not in os.environ.get("PATH", ""):
        os.environ["PATH"] = f"{user_bin}:{os.environ.get('PATH','')}"
    return user_site, user_bin

def is_module_available(module_name):
    try:
        return importlib.util.find_spec(module_name) is not None
    except ModuleNotFoundError:
        return False

def install_if_missing(pip_name, import_name=None):
    """DRIVER install: pip install --user only if the import is missing."""
    import_name = import_name or pip_name
    if not is_module_available(import_name):
        print(f"Installing {pip_name} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--user", pip_name])
    else:
        print(f"{pip_name} already installed.")

def bootstrap_driver():
    """Install the full driver-side stack from the manifest (one driver path)."""
    _user_site, _user_bin = ensure_user_site()
    log(f"User site-packages: {_user_site}")
    log(f"User bin: {_user_bin}")
    for pip_name, import_name in PACKAGES["driver"]:
        install_if_missing(pip_name, import_name)
    log("✓ All driver packages installed/verified")

log_section("DRIVER BOOTSTRAP")
bootstrap_driver()


DRIVER BOOTSTRAP
User site-packages: /home/spark/.local/lib/python3.10/site-packages
User bin: /home/spark/.local/bin
keyring already installed.
ipython-secrets already installed.
Installing oauth2client ...
pyarrow already installed.
fsspec already installed.
s3fs already installed.
scipy already installed.
Installing statsmodels ...
Installing matplotlib ...
scikit-learn already installed.
keyrings.cryptfile already installed.
Installing bokeh==3.6.2 ...
Installing jupyter_bokeh ...
Installing panel==1.5.2 ...
Installing holoviews==1.19.0 ...
Installing hvplot==0.10.0 ...
Installing datashader==0.16.3 ...
Installing dask[dataframe]==2024.9.1 ...
Installing distributed==2024.9.1 ...
Installing reportlab ...
Installing prophet ...
openpyxl already installed.
tqdm already installed.
✓ All driver packages installed/verified


In [14]:
# SPARKCASTER DISTRIBUTED PROCESSING
# Using SparkCaster for distributed execution across the cluster
# No multiprocessing needed - Spark handles task distribution

print("✓ Using SparkCaster for distributed processing")
print("  Tasks will be distributed across all Spark executors")
print("  No single-node memory limits or CPU constraints")

✓ Using SparkCaster for distributed processing
  Tasks will be distributed across all Spark executors
  No single-node memory limits or CPU constraints


In [15]:
# ── Credentials: initialize the encrypted keyring (shared by CAIOS) ──────────
import keyring, os
from getpass import getpass
from keyrings.cryptfile.cryptfile import CryptFileKeyring
from pathlib import Path

_KEYRING_READY = False

def ensure_keyring():
    """Initialize the CryptFile keyring exactly once per kernel (idempotent).

    The CAIOS credentials cell calls this; the master password is only
    prompted the first time.
    """
    global _KEYRING_READY
    if _KEYRING_READY:
        return
    os.environ["KEYRING_CRYPTFILE_PATH"] = f"{Path.home()}/.local/share/python_keyring/cryptfile_pass.cfg"
    kr = CryptFileKeyring()
    kr.keyring_key = getpass("Set/enter master password for encrypted keyring: ")
    keyring.set_keyring(kr)
    _KEYRING_READY = True

ensure_keyring()


In [16]:
# Common imports
import pandas as pd
import numpy as np


In [17]:
# ── CAIOS credentials (reuses the keyring initialized above) ────────────────
ensure_keyring()   # no-op / no re-prompt if already initialized this kernel

caios_access_key = keyring.get_password("caios", "access_key")
caios_secret_key = keyring.get_password("caios", "secret_key")
if not caios_access_key:
    caios_access_key = input("Enter CAIOS access key: ")
    keyring.set_password("caios", "access_key", caios_access_key)
if not caios_secret_key:
    caios_secret_key = getpass("Enter CAIOS secret key: ")
    keyring.set_password("caios", "secret_key", caios_secret_key)

log("✓ CAIOS credentials configured")


✓ CAIOS credentials configured


In [18]:
#temp

import os, subprocess
print("cwd:", os.getcwd())
print("HOME:", os.path.expanduser("~"))
print(subprocess.run(["bash","-lc","df -h | grep -iE 'mnt|shared|home|pvc|workspace|jbok' || true"],
                     capture_output=True, text=True).stdout)
print("roots:", [d for d in os.listdir('/') if not d.startswith('.')])


cwd: /opt/spark/work-dir
HOME: /home/spark

roots: ['bin', 'boot', 'dev', 'etc', 'home', 'lib', 'lib32', 'lib64', 'libx32', 'media', 'mnt', 'opt', 'proc', 'root', 'run', 'sbin', 'srv', 'sys', 'tmp', 'usr', 'var', '__cacert_entrypoint.sh']


In [19]:
#pull in data with CLUSTER resources (not local mode!)
# 
from spark.nessie import NessieSparkClient
from pyspark.sql import SparkSession
import sys
import site
import os

# Get user site-packages path
user_site = site.getusersitepackages()
print(f"User site-packages: {user_site}")

# Configure Spark to use cluster resources AND install packages on executors
spark = SparkSession.builder \
    .appName("NessieTimeSeriesForecast") \
    .config("spark.executor.memory", "16g") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.cores", "4") \
    .config("spark.executor.instances", "20") \
    .config("spark.sql.shuffle.partitions", "400") \
    .config("spark.driver.maxResultSize", "4g") \
    .config("spark.sql.execution.arrow.maxRecordsPerBatch", "10000") \
    .config("spark.dynamicAllocation.enabled", "true") \
    .config("spark.dynamicAllocation.minExecutors", "5") \
    .config("spark.dynamicAllocation.maxExecutors", "25") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .config("spark.pyspark.python", sys.executable) \
    .config("spark.pyspark.driver.python", sys.executable) \
    .getOrCreate()

print("="*60)
print("SPARK CLUSTER CONFIGURATION")
print("="*60)
print(f"Executor Memory: {spark.conf.get('spark.executor.memory')}")
print(f"Driver Memory: {spark.conf.get('spark.driver.memory')}")
print(f"Executor Cores: {spark.conf.get('spark.executor.cores')}")
print(f"Executor Instances: {spark.conf.get('spark.executor.instances')}")
print(f"Dynamic Allocation: {spark.conf.get('spark.dynamicAllocation.enabled')}")
print(f"Min/Max Executors: {spark.conf.get('spark.dynamicAllocation.minExecutors')}/{spark.conf.get('spark.dynamicAllocation.maxExecutors')}")
print("="*60)

# Executor dependencies are handled elsewhere, NOT here:
#   * wheelhouse build cell -> downloads the offline wheels
#   * forecasting cell -> installs them on each executor at task runtime
# (Driver-side online installs must not be pushed to workers, which have no internet.)
    
# Set up Nessie Spark client
ness = NessieSparkClient(
    svc_url="http://kf-proxy.nessie.svc.cluster.local:19120/api/v2",
    nessie_endpoint="http://nessie-prd.cwobject.com",
    caios_access_key=caios_access_key,
    caios_secret_key=caios_secret_key,
    dbtcaster=True,
)
# Turn off warnings
spark.sparkContext.setLogLevel("ERROR")

# Load data from Nessie catalog
df = ness.sql("select * from sandbox.sandbox_finance.dcgm_metrics_summary_imputed")
df.show(5, truncate=False)
print(f"\nTotal rows: {df.count():,}")

User site-packages: /home/spark/.local/lib/python3.10/site-packages
SPARK CLUSTER CONFIGURATION
Executor Memory: 16g
Driver Memory: 8g
Executor Cores: 4
Executor Instances: 20
Dynamic Allocation: true
Min/Max Executors: 5/25
+-------------------+----------+----------------+-----------------+---------------------+--------------------+-----------------------+---------------------+-----------------------+---------------+------------+------------+------------+-------------------+-------------------+------------------+------------------+-----------------+-----------------+-----------------+-----------------+-----------------+---------------+-------------------+------------------+-----------------+-----------------+-----------------+--------------+--------------+--------------+-------------+-------------------+-------------------+------------+------------+------------+-------------------+-------------------+------------------+-----------------+------------------+------------------+----------

In [20]:
# BUILD WHEELHOUSE FOR EXECUTORS (no internet needed on workers)
import os, sys, subprocess, shutil
from pyspark import SparkFiles

wheel_dir = '/tmp/sparkcaster_wheels'
zip_base = '/tmp/sparkcaster_wheels'
zip_path = f'{zip_base}.zip'

if not os.path.exists(zip_path):
    os.makedirs(wheel_dir, exist_ok=True)
    # Download wheels on driver (executors will install from this zip)
    packages = PACKAGES['executor']   # single manifest, shared with the executor runtime install
    subprocess.check_call([sys.executable, '-m', 'pip', 'download', '-d', wheel_dir] + packages)
    shutil.make_archive(zip_base, 'zip', wheel_dir)

spark.sparkContext.addFile(zip_path)
print(f'✓ Distributed wheelhouse: {zip_path}')


✓ Distributed wheelhouse: /tmp/sparkcaster_wheels.zip


In [21]:
# Keep data as Spark DataFrame for distributed processing
from pyspark.sql.functions import col, when, to_date
from pyspark.sql import functions as F
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

print("Preparing Spark DataFrame for distributed forecasting...")

# Convert day column to date type (if not already)
df = df.withColumn('day', to_date(col('day')))

# Create region_summary field using Spark operations
# Logic: if region starts with 'EU' then 'EU', else 'NAM'
df = df.withColumn('region_summary', 
                   when(col('region').startswith('EU'), 'EU')
                   .otherwise('NAM'))

# Cache the DataFrame for faster access during distributed processing
df = df.cache()

# Get basic statistics (collect only summary info, not full data)
row_count = df.count()
date_range = df.select(F.min('day'), F.max('day')).first()
columns = df.columns

print(f"Data shape: {row_count} rows × {len(columns)} columns")
print(f"Date range: {date_range[0]} to {date_range[1]}")
print(f"\nColumns: {columns}")
print(f"\nRegion Summary Distribution:")
df.groupBy('region_summary').count().show()
print(f"\nOriginal Regions → Region Summary Mapping:")
df.select('region', 'region_summary').distinct().orderBy('region').show()
print(f"\nSample data (first 5 rows):")
df.show(5)

print(f"\n✓ Data ready for SparkCaster distributed forecasting")
print(f"  DataFrame cached with {row_count:,} rows")

Preparing Spark DataFrame for distributed forecasting...
Data shape: 110240 rows × 49 columns
Date range: 2025-01-23 to 2026-05-31

Columns: ['day', 'region', 'is_training_norm', 'is_multinode_norm', 'product_resolved_norm', 'product_segment_norm', 'gpu_count_expected_norm', 'customer_segment_norm', 'customer_name_norm', 'peak_power_unit', 'gpu_util_p50', 'gpu_util_p95', 'gpu_util_p99', 'tensor_util_p50', 'tensor_util_p95', 'tensor_util_p99', 'chip_power_p50', 'chip_power_p95', 'chip_power_p99', 'redfish_power_p50', 'redfish_power_p95', 'redfish_power_p99', 'dram_active_p50', 'dram_active_p95', 'dram_active_p99', 'mem_copy_util_p50', 'mem_copy_util_p95', 'mem_copy_util_p99', 'vram_usage_p50', 'vram_usage_p95', 'vram_usage_p99', 'sm_active_p50', 'sm_active_p95', 'sm_active_p99', 'sm_clock_p50', 'sm_clock_p95', 'sm_clock_p99', 'sm_occupancy_p50', 'sm_occupancy_p95', 'sm_occupancy_p99', 'tflops_avg', 'tflops_p50', 'tflops_p95', 'tflops_p99', 'node_count_daily_avg', 'tflops_node_avg_p50', 

In [22]:
# Define metrics to forecast
# Note: Using actual column names from dcgm_metrics_summary_imputed table
METRICS = [
    'gpu_util_p50',
    'tensor_util_p50', 
    'tensor_util_p95', 
    'tensor_util_p99',
    'chip_power_p50', 
    'chip_power_p95',
    'redfish_power_p50', 
    'redfish_power_p95',
    # Note: tflops columns are named differently in this table
    # 'tflops_total_p50',  # doesn't exist - use tflops_p50
    # 'tflops_total_p95',  # doesn't exist - use tflops_p95
    # 'tflops_node_avg_p50',  # need to verify actual column name
    # 'tflops_node_avg_p95',  # need to verify actual column name
    # 'tflops_node_avg_p99'   # need to verify actual column name
]

print(f"⚠️  WARNING: Some tflops metrics commented out - need to verify column names")
print(f"   Available tflops columns in error message: tflops_p50, tflops_p95")
print(f"   You may want to check df.columns to see all available metrics\n")

# Define grouping columns (using _norm suffix for normalized columns)
# Using region_summary instead of region (EU vs NAM)
GROUPINGS = {
    'All': [],
    'product_resolved': ['product_resolved_norm'],
    'product_segment': ['product_segment_norm'],
    'customer_segment': ['customer_segment_norm'],
    'region_summary': ['region_summary'],
    'region_summary+product_segment': ['region_summary', 'product_segment_norm'],  # Combined grouping
    'region_summary+product_resolved': ['region_summary', 'product_resolved_norm'],  # Combined grouping
    'product_segment+product_resolved': ['product_segment_norm', 'product_resolved_norm']
}

print(f"Metrics to forecast: {len(METRICS)}")
print(f"Grouping strategies: {list(GROUPINGS.keys())}")
print(f"\nGrouping details:")
print(f"  - All: Global aggregate")
print(f"  - product_resolved: By GPU type (H100, H200, L40, etc.)")
print(f"  - product_segment: By segment (HGX, PCIE)")
print(f"  - customer_segment: By customer type")
print(f"  - region_summary: By region (EU vs NAM)")
print(f"  - region_summary+product_segment: By region AND segment (e.g., EU-HGX, NAM-PCIE)")
print(f"  - region_summary+product_resolved: By region AND product (e.g., EU-B200, NAM-GB200)")
print(f"\nTotal combinations: {len(METRICS)} metrics × {len(GROUPINGS)} groupings = {len(METRICS) * len(GROUPINGS)} series")

print(f"\n💡 TIP: Run df.columns to see all available column names if you want to add more metrics")

⚠️  WARNING: Some tflops metrics commented out - need to verify column names
   Available tflops columns in error message: tflops_p50, tflops_p95
   You may want to check df.columns to see all available metrics

Metrics to forecast: 8
Grouping strategies: ['All', 'product_resolved', 'product_segment', 'customer_segment', 'region_summary', 'region_summary+product_segment', 'region_summary+product_resolved', 'product_segment+product_resolved']

Grouping details:
  - All: Global aggregate
  - product_resolved: By GPU type (H100, H200, L40, etc.)
  - product_segment: By segment (HGX, PCIE)
  - customer_segment: By customer type
  - region_summary: By region (EU vs NAM)
  - region_summary+product_segment: By region AND segment (e.g., EU-HGX, NAM-PCIE)
  - region_summary+product_resolved: By region AND product (e.g., EU-B200, NAM-GB200)

Total combinations: 8 metrics × 8 groupings = 64 series

💡 TIP: Run df.columns to see all available column names if you want to add more metrics


In [23]:
# Data preprocessing - Create Spark DataFrame with all metric/grouping combinations
from pyspark.sql.functions import col, concat_ws, avg, collect_list, struct
from pyspark.sql import Window

print("Preparing time series combinations for distributed processing...")

# We'll create a DataFrame where each row represents a metric/grouping/group_key combination
# This will be partitioned and distributed to executors via SparkCaster

def create_time_series_tasks(spark_df, metrics, groupings):
    """
    Create a Spark DataFrame where each row represents one time series forecasting task
    
    Returns: Spark DataFrame with columns:
    - metric: the metric name
    - grouping_name: the grouping strategy name  
    - group_key: the specific group identifier
    - time_series_data: array of structs with (day, value)
    """
    from pyspark.sql.functions import lit, collect_list, struct, concat_ws
    
    tasks = []
    
    # For each metric and grouping combination
    for metric in metrics:
        for grouping_name, group_cols in groupings.items():
            
            if len(group_cols) == 0:
                # 'All' grouping - aggregate everything by day
                agg_df = spark_df.groupBy('day').agg(
                    avg(metric).alias('value')
                ).withColumn('group_key', lit('All'))
                
            else:
                # Group by specified columns + day
                agg_df = spark_df.groupBy(*(group_cols + ['day'])).agg(
                    avg(metric).alias('value')
                ).withColumn('group_key', concat_ws('_', *group_cols))
            
            # Add metadata columns and select ONLY the columns we need (consistent schema)
            agg_df = agg_df.select(
                lit(metric).alias('metric'),
                lit(grouping_name).alias('grouping_name'),
                col('group_key'),
                col('day'),
                col('value')
            )
            
            tasks.append(agg_df)
    
    # Union all tasks - now they all have the same 5 columns
    from functools import reduce
    all_tasks = reduce(lambda df1, df2: df1.union(df2), tasks)
    
    # Group by metric/grouping/group_key to create array of (day, value) pairs
    result = all_tasks.groupBy('metric', 'grouping_name', 'group_key').agg(
        collect_list(struct('day', 'value')).alias('time_series_data')
    )
    
    return result

print("✓ Data preparation function created")
print("  This will create one task per metric/grouping/group_key combination")

Preparing time series combinations for distributed processing...
✓ Data preparation function created
  This will create one task per metric/grouping/group_key combination


In [24]:
# ── EXECUTOR FORECASTING (runs on Spark workers) ─────────────────────────────
# One task per (metric, grouping, group_key). Fits 5 models, picks the best by
# MAE, and returns the CANONICAL result schema (documented in the function).
# Prediction intervals (P10/P90) and [0,1] utilization bounds are computed HERE
# so the distributed path produces exactly what plotting/export consume.
import time, json
import pandas as pd
import numpy as np

# --- Single source of truth for split / horizon (CONFIG mirrors these) -------
TRAIN_SPLIT = 0.7
FORECAST_DAYS = 1100
_Z_P10_P90 = 1.2816            # 10th/90th percentile of a normal (80% central band)
EXECUTOR_PACKAGES = PACKAGES["executor"]   # from the dependency manifest (cell 1)


def forecast_time_series_row(row):
    """Fit models for one time series on an executor; return a JSON result.

    CANONICAL RESULT SCHEMA (the single contract shared by forecasting,
    plotting, and export):
      { metric, grouping, grouping_name, group_key,
        status: 'completed'|'skipped'|'error',
        best_model, train_size, test_size,
        train_dates[], test_dates[], train_values[], test_values[],
        results: { <model>: {
            status: 'success'|'failed',
            metrics: {MSE, RMSE, MAPE, MAE},
            mae,                    # convenience copy for best-model ranking
            test_predictions[],     # aligned to test window
            forecast[],             # P50 point forecast, length FORECAST_DAYS
            forecast_lower[],       # P10
            forecast_upper[],       # P90
            fitted[],               # in-sample fit, aligned to train (may be [])
        } } }
    """
    import subprocess, sys, warnings, importlib, os
    warnings.filterwarnings('ignore')

    # --- Executor runtime dependency ensure: OFFLINE install from wheelhouse ---
    # WHY separate from the driver: workers have no internet, so we install the
    # minimal manifest from the wheelhouse zip shipped via SparkFiles.
    need = []
    for mod in ('statsmodels', 'scipy'):
        try:
            importlib.import_module(mod)
        except ImportError:
            need.append(mod)
    if need:
        try:
            import zipfile, tempfile
            from pyspark import SparkFiles
            wheel_zip = SparkFiles.get('sparkcaster_wheels.zip')
            extract_dir = os.path.join(tempfile.gettempdir(), 'sparkcaster_wheels')
            if not os.path.exists(extract_dir):
                os.makedirs(extract_dir, exist_ok=True)
                with zipfile.ZipFile(wheel_zip, 'r') as zf:
                    zf.extractall(extract_dir)
            subprocess.check_call(
                [sys.executable, '-m', 'pip', 'install', '--user', '--no-index',
                 '--find-links', extract_dir] + list(EXECUTOR_PACKAGES),
                stderr=subprocess.DEVNULL, stdout=subprocess.DEVNULL, timeout=180)
            importlib.reload(importlib.import_module('site'))
        except Exception as install_error:
            return json.dumps({
                'metric': getattr(row, 'metric', 'unknown'),
                'grouping': getattr(row, 'grouping_name', 'unknown'),
                'grouping_name': getattr(row, 'grouping_name', 'unknown'),
                'group_key': getattr(row, 'group_key', 'unknown'),
                'status': 'error',
                'error': f'Executor package install failed: {str(install_error)[:150]}',
            })

    import numpy as np
    import pandas as pd

    metric_name = getattr(row, 'metric', 'unknown')
    grouping_name = getattr(row, 'grouping_name', 'unknown')
    group_key = getattr(row, 'group_key', 'unknown')

    def _err(msg):
        return json.dumps({'metric': metric_name, 'grouping': grouping_name,
                           'grouping_name': grouping_name, 'group_key': group_key,
                           'status': 'error', 'error': str(msg)[:200]})

    def _skip(reason, n):
        return json.dumps({'metric': metric_name, 'grouping': grouping_name,
                           'grouping_name': grouping_name, 'group_key': group_key,
                           'status': 'skipped', 'reason': reason, 'data_points': n})

    try:
        from statsmodels.tsa.holtwinters import ExponentialSmoothing
        from statsmodels.tsa.arima.model import ARIMA
        from statsmodels.tsa.statespace.sarimax import SARIMAX
        try:
            from prophet import Prophet
            has_prophet = True
        except Exception:
            has_prophet = False

        ts = pd.DataFrame([
            {'day': pd.to_datetime(p.day),
             'value': float(p.value) if p.value is not None else 0.0}
            for p in row.time_series_data
        ])
        if len(ts) < 21:
            return _skip('insufficient_data', len(ts))
        ts = ts.dropna().sort_values('day').reset_index(drop=True)
        if len(ts) < 21:
            return _skip('insufficient_data_after_cleaning', len(ts))

        split_idx = int(len(ts) * TRAIN_SPLIT)
        train = ts.iloc[:split_idx].copy()
        test = ts.iloc[split_idx:].copy()
        train_vals = train['value'].to_numpy(dtype=float)
        test_vals = test['value'].to_numpy(dtype=float)
        n_test = len(test_vals)
        horizon = n_test + FORECAST_DAYS

        util = any(k in metric_name.lower()
                   for k in ('util', 'utilization', 'usage', 'saturation'))

        def _clip(a):
            a = np.asarray(a, dtype=float)
            a = np.maximum(a, 0.0)               # metrics are non-negative
            if util:
                a = np.minimum(a, 1.0)           # utilization bounded to [0, 1]
            return a

        def _metrics(actual, pred):
            actual = np.asarray(actual, float); pred = np.asarray(pred, float)
            mse = float(np.mean((actual - pred) ** 2))
            rmse = float(np.sqrt(mse))
            denom = np.where(actual == 0, np.nan, actual)
            mape = float(np.nanmean(np.abs((actual - pred) / denom)) * 100)
            mae = float(np.mean(np.abs(actual - pred)))
            return {'MSE': mse, 'RMSE': rmse, 'MAPE': mape, 'MAE': mae}

        def _pack(full_forecast, fitted_full):
            """Split a length-`horizon` forecast into test + future and build the
            canonical per-model dict. Intervals are an empirical residual band."""
            full_forecast = np.asarray(full_forecast, float)
            test_pred = full_forecast[:n_test]
            future = full_forecast[n_test:n_test + FORECAST_DAYS]
            if fitted_full is not None and len(fitted_full) == len(train_vals):
                resid = train_vals - np.asarray(fitted_full, float)
            else:
                resid = test_vals - test_pred[:len(test_vals)]
            sigma = float(np.nanstd(resid)) if len(resid) else 0.0
            band = _Z_P10_P90 * sigma
            m = _metrics(test_vals, test_pred)
            return {
                'status': 'success',
                'metrics': m,
                'mae': m['MAE'],
                'test_predictions': _clip(test_pred).tolist(),
                'forecast': _clip(future).tolist(),
                'forecast_lower': _clip(future - band).tolist(),
                'forecast_upper': _clip(future + band).tolist(),
                'fitted': (_clip(fitted_full).tolist() if fitted_full is not None else []),
            }

        results = {}

        # Model 1: Exponential Smoothing (additive trend + seasonal)
        try:
            fit = ExponentialSmoothing(train_vals, seasonal_periods=7,
                                       trend='add', seasonal='add').fit()
            fc = np.asarray(fit.forecast(steps=horizon), float)
            results['exponential_smoothing'] = _pack(fc, getattr(fit, 'fittedvalues', None))
        except Exception as e:
            results['exponential_smoothing'] = {'status': 'failed', 'error': str(e)[:100]}

        # Model 2: ARIMA
        try:
            fit = ARIMA(train_vals, order=(5, 1, 0)).fit()
            fc = np.asarray(fit.forecast(steps=horizon), float)
            results['arima'] = _pack(fc, getattr(fit, 'fittedvalues', None))
        except Exception as e:
            results['arima'] = {'status': 'failed', 'error': str(e)[:100]}

        # Model 3: SARIMA (weekly seasonality)
        try:
            fit = SARIMAX(train_vals, order=(1, 1, 1),
                          seasonal_order=(1, 1, 1, 7)).fit(disp=False)
            fc = np.asarray(fit.forecast(steps=horizon), float)
            results['sarima'] = _pack(fc, getattr(fit, 'fittedvalues', None))
        except Exception as e:
            results['sarima'] = {'status': 'failed', 'error': str(e)[:100]}

        # Model 4: Prophet (uses its NATIVE 80% intervals when available)
        if has_prophet:
            try:
                pdf = train.rename(columns={'day': 'ds', 'value': 'y'})
                m = Prophet(daily_seasonality=True, weekly_seasonality=True,
                            yearly_seasonality=False, interval_width=0.8)
                m.fit(pdf)
                fdf = m.make_future_dataframe(periods=horizon)
                pred = m.predict(fdf)
                yhat = pred['yhat'].to_numpy()
                lo = pred['yhat_lower'].to_numpy()
                hi = pred['yhat_upper'].to_numpy()
                ntr = len(train_vals)
                fitted_full = yhat[:ntr]
                test_pred = yhat[ntr:ntr + n_test]
                future = yhat[ntr + n_test:ntr + horizon]
                low = lo[ntr + n_test:ntr + horizon]
                up = hi[ntr + n_test:ntr + horizon]
                mm = _metrics(test_vals, test_pred)
                results['prophet'] = {
                    'status': 'success', 'metrics': mm, 'mae': mm['MAE'],
                    'test_predictions': _clip(test_pred).tolist(),
                    'forecast': _clip(future).tolist(),
                    'forecast_lower': _clip(low).tolist(),
                    'forecast_upper': _clip(up).tolist(),
                    'fitted': _clip(fitted_full).tolist(),
                }
            except Exception as e:
                results['prophet'] = {'status': 'failed', 'error': str(e)[:100]}

        # Model 5: Holt-Winters (damped, multiplicative seasonal)
        try:
            fit = ExponentialSmoothing(train_vals, seasonal_periods=7, trend='add',
                                       seasonal='mul', damped_trend=True).fit()
            fc = np.asarray(fit.forecast(steps=horizon), float)
            results['holt_winters'] = _pack(fc, getattr(fit, 'fittedvalues', None))
        except Exception as e:
            results['holt_winters'] = {'status': 'failed', 'error': str(e)[:100]}

        ok = {k: v for k, v in results.items() if v.get('status') == 'success'}
        best_model = min(ok.items(), key=lambda kv: kv[1]['mae'])[0] if ok else None

        return json.dumps({
            'metric': metric_name, 'grouping': grouping_name, 'grouping_name': grouping_name,
            'group_key': group_key, 'status': 'completed', 'best_model': best_model,
            'results': results, 'train_size': len(train), 'test_size': len(test),
            'train_dates': train['day'].astype(str).tolist(),
            'test_dates': test['day'].astype(str).tolist(),
            'train_values': train['value'].tolist(),
            'test_values': test['value'].tolist(),
        })

    except ImportError as ie:
        return _err(f'Import failed after install: {ie}')
    except Exception as e:
        return _err(e)


print("✓ Executor forecasting defined (P10/P90 intervals + [0,1] util bounds, canonical schema)")


✓ Executor forecasting defined (P10/P90 intervals + [0,1] util bounds, canonical schema)


In [25]:
# SPARK DISTRIBUTED ORCHESTRATION - Run distributed forecasting
from pyspark.sql.types import StringType
from pyspark.sql.functions import udf
import time
import json

def run_sparkcaster_forecasting(spark_df, metrics, groupings):
    """
    Run time series forecasting using Spark distributed processing

    Parameters:
    - spark_df: Spark DataFrame with time series data
    - metrics: List of metrics to forecast
    - groupings: Dictionary of grouping strategies

    Returns: List of dict results
    """

    print(f"{'='*80}")
    print(f"SPARK DISTRIBUTED TIME SERIES FORECASTING")
    print(f"{'='*80}")
    print(f"Metrics: {len(metrics)}")
    print(f"Groupings: {len(groupings)}")
    print(f"Cluster: 20 executors × 4 cores = 80 parallel workers")
    print(f"{'='*80}")

    start_time = time.time()

    # Step 1: Create tasks DataFrame (one row per metric/grouping/group combination)
    print("Step 1: Creating time series tasks...")
    tasks_df = create_time_series_tasks(spark_df, metrics, groupings)

    # Count tasks
    n_tasks = tasks_df.count()
    print(f"  Created {n_tasks} forecasting tasks")

    # Step 2: Repartition for distributed processing
    print(f"Step 2: Repartitioning for distributed processing...")
    n_partitions = min(n_tasks, 200)  # Max 200 partitions
    tasks_df = tasks_df.repartition(n_partitions)
    print(f"  Using {n_partitions} partitions")

    # Step 3: Apply distributed processing using RDD map
    print(f"Step 3: Distributing forecast function to executors...")
    print(f"Step 4: Running distributed forecasting...")
    print(f"  Processing {n_tasks} tasks across cluster...")

    # Use RDD map to distribute the work
    results_rdd = tasks_df.rdd.map(forecast_time_series_row)

    # Collect results
    print(f"Step 5: Collecting results...")
    results_list = results_rdd.collect()

    # Parse JSON results
    parsed_results = [json.loads(r) for r in results_list]

    end_time = time.time()
    duration = end_time - start_time

    print(f"{'='*80}")
    print(f"FORECASTING COMPLETE")
    print(f"{'='*80}")
    print(f"Total tasks: {len(parsed_results)}")
    print(f"Duration: {duration:.1f} seconds ({duration/60:.1f} minutes)")
    print(f"Throughput: {len(parsed_results) / duration:.1f} tasks/second")
    print(f"{'='*80}")

    return parsed_results

print("✓ Spark distributed orchestration function ready")

✓ Spark distributed orchestration function ready


In [26]:
# CONFIGURATION
# SparkCaster handles all distribution automatically

CONFIG = {
    'train_split': TRAIN_SPLIT,     # single source of truth (defined in the executor forecasting cell)
    'forecast_days': FORECAST_DAYS, # single source of truth (defined in the executor forecasting cell)
    'n_workers': 30,                # only used by the (legacy) PERFORMANCE ANALYSIS efficiency estimate
}

print("✓ Configuration set")
print(f"  Train/Test split: {CONFIG['train_split']}")
print(f"  Forecast horizon: {CONFIG['forecast_days']} days")

✓ Configuration set
  Train/Test split: 0.7
  Forecast horizon: 1100 days


In [27]:
# Note: Plot generation happens after results are collected
# No need for separate parallel plot generation with SparkCaster

print("✓ Plots will be generated from results after forecasting completes")

✓ Plots will be generated from results after forecasting completes


In [28]:
# SPARKCASTER MEMORY MANAGEMENT
# SparkCaster automatically handles memory across executors
# No need for manual chunking or memory optimization

print("✓ Memory managed automatically by Spark cluster")
print("  Each executor has 16GB RAM")
print("  No single-node memory bottlenecks")

✓ Memory managed automatically by Spark cluster
  Each executor has 16GB RAM
  No single-node memory bottlenecks


In [29]:
# SPARKCASTER IS NOW THE DEFAULT!
# This notebook uses SparkCaster for distributed processing
# No need for alternative implementations

print("✓ SparkCaster distributed processing is the default method")
print("  All forecasting tasks distributed across Spark cluster")

✓ SparkCaster distributed processing is the default method
  All forecasting tasks distributed across Spark cluster


In [30]:
# QUICK DIAGNOSTIC: Estimate workload before running
def estimate_workload():
    """
    Quick diagnostic to estimate processing time and resource needs
    """
    log_section("WORKLOAD ESTIMATION")
    
    # Estimate series count based on metrics and groupings
    # This is an approximation - actual count will be determined when tasks are created
    estimated_series = len(METRICS) * len(GROUPINGS)
    
    # For more detailed groupings, multiply by average groups per grouping
    # Rough estimates: product_resolved ~10, customer_segment ~5, region ~2, etc.
    avg_groups_per_grouping = 5
    estimated_series = estimated_series * avg_groups_per_grouping
    
    # Estimate processing time with SparkCaster
    # Assumptions: ~30 seconds per series, distributed across 80 workers
    est_time_per_series = 30  # seconds
    est_sequential_time = estimated_series * est_time_per_series
    
    # With 80 parallel workers on Spark cluster
    n_workers = 80  # 20 executors × 4 cores
    est_parallel_time = (est_sequential_time / n_workers) * 1.5  # 1.5x overhead for coordination
    
    log(f"\nDataset Analysis:")
    log(f"  Metrics: {len(METRICS)}")
    log(f"  Grouping strategies: {len(GROUPINGS)}")
    log(f"  Estimated time series: ~{estimated_series} (rough estimate)")
    log(f"  Models per series: 5")
    log(f"  Total model runs: ~{estimated_series * 5}")
    
    log(f"\nTime Estimates:")
    log(f"  Sequential processing: ~{est_sequential_time/60:.1f} minutes")
    log(f"  SparkCaster ({n_workers} workers): ~{est_parallel_time/60:.1f} minutes")
    log(f"  Speedup: ~{n_workers}x")
    
    log(f"\nCluster Resources:")
    log(f"  Executors: 20")
    log(f"  Cores per executor: 4")
    log(f"  Total parallel workers: {n_workers}")
    log(f"  Memory per executor: 16GB")
    
    log(f"\nRecommendation:")
    if estimated_series < 100:
        log(f"  ✓ Small dataset - SparkCaster will complete quickly")
        log(f"  ✓ Expected completion: {est_parallel_time/60:.1f} minutes")
    elif estimated_series < 500:
        log(f"  ✓ Medium dataset - perfect for SparkCaster")
        log(f"  ✓ Expected completion: {est_parallel_time/60:.1f} minutes")
    elif estimated_series < 2000:
        log(f"  ⚡ Large dataset - SparkCaster will handle this efficiently")
        log(f"  ⚡ Expected completion: {est_parallel_time/60:.1f} minutes")
    else:
        log(f"  ⚡⚡ Very large dataset - SparkCaster scales linearly")
        log(f"  ⚡⚡ Expected completion: {est_parallel_time/60:.1f} minutes")
        log(f"  💡 Tip: Monitor Spark UI for task progress")
    
    log("="*80)
    
    return estimated_series, est_parallel_time

# Run estimation
log("\nRunning workload estimation...\n")
estimated_series, estimated_time = estimate_workload()
log(f"\nProceed to next cell to start forecasting!")


Running workload estimation...

WORKLOAD ESTIMATION

Dataset Analysis:
  Metrics: 8
  Grouping strategies: 8
  Estimated time series: ~320 (rough estimate)
  Models per series: 5
  Total model runs: ~1600

Time Estimates:
  Sequential processing: ~160.0 minutes
  SparkCaster (80 workers): ~3.0 minutes
  Speedup: ~80x

Cluster Resources:
  Executors: 20
  Cores per executor: 4
  Total parallel workers: 80
  Memory per executor: 16GB

Recommendation:
  ✓ Medium dataset - perfect for SparkCaster
  ✓ Expected completion: 3.0 minutes

Proceed to next cell to start forecasting!


In [31]:
# RUN SPARKCASTER DISTRIBUTED FORECASTING
log_section("STARTING SPARKCASTER DISTRIBUTED FORECASTING")
log(f"Cluster: 20 executors × 4 cores = 80 parallel workers")
log("Estimated speedup: 50-100x faster than single-node processing!")
log("="*80)
log("")

# Run SparkCaster forecasting
import time
_forecast_start = time.time()
all_results = run_sparkcaster_forecasting(df, METRICS, GROUPINGS)
forecast_time = time.time() - _forecast_start  # wall-clock seconds; used by the PERFORMANCE ANALYSIS cell
all_results_df = pd.DataFrame(all_results)

log(f"✓ Forecasting complete!")
log(f"  Total results: {len(all_results_df)}")
log(f"  Successful: {len(all_results_df[all_results_df['status'] == 'completed'])}")
log(f"  Skipped: {len(all_results_df[all_results_df['status'] == 'skipped'])}")
log(f"  Errors: {len(all_results_df[all_results_df['status'] == 'error'])}")

# Build plot_data_list and all_plots for downstream cells
plot_data_list = []
all_plots = []

for result in all_results:
    if result.get('status') != 'completed':
        continue

    best_model = result.get('best_model')
    grouping = result.get('grouping') or result.get('grouping_name')

    metadata = {
        'metric': result.get('metric'),
        'grouping': grouping,
        'group_key': result.get('group_key'),
        'train_dates': pd.to_datetime(result.get('train_dates', [])),
        'test_dates': pd.to_datetime(result.get('test_dates', [])),
        'train_values': pd.Series(result.get('train_values', [])),
        'test_values': pd.Series(result.get('test_values', []))
    }

    results = {}
    for model_name, model_result in (result.get('results') or {}).items():
        if model_result.get('status') != 'success':
            continue

        results[model_name] = {
            'test_predictions': pd.Series(model_result.get('test_predictions', [])),
            'forecast': pd.Series(model_result.get('forecast', [])),
            'forecast_lower': pd.Series(model_result.get('forecast_lower', [])),
            'forecast_upper': pd.Series(model_result.get('forecast_upper', [])),
            'fitted': pd.Series(model_result.get('fitted', [])) if model_result.get('fitted') is not None else None,
            'metrics': model_result.get('metrics', {}),
            'metadata': metadata
        }

    if not results or best_model not in results:
        continue

    plot_entry = {
        'metric': result.get('metric'),
        'grouping': grouping,
        'group_key': result.get('group_key'),
        'best_model': best_model,
        'results': results,
    }

    plot_data_list.append(plot_entry)
    all_plots.append(plot_entry)



STARTING SPARKCASTER DISTRIBUTED FORECASTING
Cluster: 20 executors × 4 cores = 80 parallel workers
Estimated speedup: 50-100x faster than single-node processing!

SPARK DISTRIBUTED TIME SERIES FORECASTING
Metrics: 8
Groupings: 8
Cluster: 20 executors × 4 cores = 80 parallel workers
Step 1: Creating time series tasks...


  Created 496 forecasting tasks
Step 2: Repartitioning for distributed processing...
  Using 200 partitions
Step 3: Distributing forecast function to executors...
Step 4: Running distributed forecasting...
  Processing 496 tasks across cluster...
Step 5: Collecting results...
FORECASTING COMPLETE
Total tasks: 496
Duration: 37.8 seconds (0.6 minutes)
Throughput: 13.1 tasks/second
✓ Forecasting complete!
  Total results: 496
  Successful: 490
  Skipped: 0
  Errors: 6


In [32]:
# Check what errors occurred
# print("Sample errors:")
# for i, result in enumerate(all_results[:5]):
#     print(f"\n=== Task {i+1} ===")
#     print(f"Metric: {result.get('metric')}")
#     print(f"Grouping: {result.get('grouping_name')}")
#     print(f"Status: {result.get('status')}")
#     print(f"Error: {result.get('error', 'N/A')}")

In [33]:
# FORECAST QUALITY DIAGNOSTIC
# Run this after forecasting completes to check if warnings are a problem

from collections import Counter

log_section("FORECAST QUALITY ANALYSIS")

# Count statuses from results
statuses = []
model_success = Counter()

for result in all_results:
    if 'best_model' in result:
        model_success[result['best_model']] += 1

# Calculate metrics
total_series = len(plot_data_list)
successful_forecasts = len([p for p in plot_data_list if p is not None])
success_rate = (successful_forecasts / total_series * 100) if total_series > 0 else 0

log(f"\nOverall Statistics:")
log(f"  Total time series: {total_series}")
log(f"  Successful forecasts: {successful_forecasts}")
log(f"  Success rate: {success_rate:.1f}%")

log(f"\nBest Model Distribution:")
for model, count in model_success.most_common():
    pct = count / successful_forecasts * 100 if successful_forecasts > 0 else 0
    log(f"  {model}: {count} ({pct:.1f}%)")

log(f"\nQuality Assessment:")
if success_rate > 90:
    log("  ✅ EXCELLENT - Warnings are normal, no action needed")
    log("     Your error handling is working perfectly")
elif success_rate > 75:
    log("  🟢 GOOD - Most series forecasting successfully")
    log("     Consider adding warning suppression for cleaner output")
elif success_rate > 60:
    log("  🟡 FAIR - Some optimization recommended")
    log("     Review failed series and consider data quality checks")
else:
    log("  🔴 POOR - Investigation needed")
    log("     Significant data quality or model configuration issues")

log("\n" + "=" * 80)


FORECAST QUALITY ANALYSIS

Overall Statistics:
  Total time series: 490
  Successful forecasts: 490
  Success rate: 100.0%

Best Model Distribution:
  arima: 271 (55.3%)
  exponential_smoothing: 84 (17.1%)
  sarima: 75 (15.3%)
  holt_winters: 60 (12.2%)

Quality Assessment:
  ✅ EXCELLENT - Warnings are normal, no action needed
     Your error handling is working perfectly



In [34]:
# ── OUTPUT HELPERS (CoreWeave Object Storage / LOTA) ─────────────────────────
# Route 1: export the final pandas DataFrames straight to CoreWeave Object
# Storage as CSV + XLSX. No local files, no Iceberg. Download from Cloud Console.
import io, os
import pandas as pd
import fsspec   # s3fs / fsspec / openpyxl come from the driver bootstrap cell

# --- Config ---
S3_BUCKET   = 'jbok-sandbox-test'
S3_ENDPOINT = 'http://cwlota.com'          # LOTA base; virtual-hosted -> http://<bucket>.cwlota.com
S3_REGION   = 'US-EAST-04A'

# One run-scoped UTC timestamp so all outputs share a single folder
from datetime import datetime, timezone
RUN_TS    = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
S3_PREFIX = f'jbok/time-series-fcst-sparkcaster/{RUN_TS}/'

# Access key / secret you created. Uses kernel env vars if set, otherwise prompts
# (getpass keeps the secret out of the notebook file). NOTE: a .env in your
# workspace is NOT readable here — this code runs in the remote Spark kernel,
# which does not mount /home/coreweave/jbok-cw.
S3_ACCESS_KEY = os.environ.get('CW_S3_ACCESS_KEY')
S3_SECRET_KEY = os.environ.get('CW_S3_SECRET_KEY')
if not S3_ACCESS_KEY or not S3_SECRET_KEY:
    import getpass
    S3_ACCESS_KEY = S3_ACCESS_KEY or getpass.getpass('CoreWeave S3 access key: ')
    S3_SECRET_KEY = S3_SECRET_KEY or getpass.getpass('CoreWeave S3 secret key: ')

STORAGE_OPTIONS = {
    'key': S3_ACCESS_KEY,
    'secret': S3_SECRET_KEY,
    'client_kwargs': {'endpoint_url': S3_ENDPOINT, 'region_name': S3_REGION},
    'config_kwargs': {'s3': {'addressing_style': 'virtual'}},   # LOTA is virtual-hosted
}

_S3_SAVED = []   # (label, s3_uri) collected across the run

def _s3_uri(name, ext):
    return f"s3://{S3_BUCKET}/{S3_PREFIX}{name}.{ext}"

def _put_bytes(uri, data):
    with fsspec.open(uri, 'wb', **STORAGE_OPTIONS) as f:
        f.write(data)

def save_df_to_s3(name, pdf, formats=('csv', 'xlsx')):
    """Write a pandas DataFrame to Object Storage as CSV and/or XLSX."""
    if pdf is None or len(pdf) == 0:
        print(f"  (skipping '{name}': empty DataFrame)")
        return
    formats = list(formats)
    if 'xlsx' in formats and len(pdf) > 1_048_575:      # Excel row limit
        print(f"  (xlsx skipped for '{name}': {len(pdf):,} rows exceed Excel's limit; CSV only)")
        formats = [f for f in formats if f != 'xlsx']
    if 'csv' in formats:
        uri = _s3_uri(name, 'csv')
        _put_bytes(uri, pdf.to_csv(index=False).encode('utf-8'))
        _S3_SAVED.append((f'{name} (csv)', uri)); print(f"  → {uri}")
    if 'xlsx' in formats:
        uri = _s3_uri(name, 'xlsx')
        buf = io.BytesIO()
        with pd.ExcelWriter(buf, engine='openpyxl') as xw:
            pdf.to_excel(xw, index=False, sheet_name=(name[:31] or 'Sheet1'))
        _put_bytes(uri, buf.getvalue())
        _S3_SAVED.append((f'{name} (xlsx)', uri)); print(f"  → {uri}")

def print_s3_manifest():
    """Print every object written this run, with its s3:// path."""
    print(f"Bucket  : {S3_BUCKET}")
    print(f"Prefix  : {S3_PREFIX}")
    print(f"Endpoint: {S3_ENDPOINT} (virtual-hosted)")
    if not _S3_SAVED:
        print("(no objects written this run)")
        return
    for label, uri in _S3_SAVED:
        print(f"  {label:52} {uri}")

print(f"✓ Object-storage export helpers ready. Target: s3://{S3_BUCKET}/{S3_PREFIX}")


✓ Object-storage export helpers ready. Target: s3://jbok-sandbox-test/jbok/time-series-fcst-sparkcaster/20260723_194659/


In [35]:
# Create and save results to Excel files

# 1. Create DataFrame for ALL models (one row per model per time series)
if len(all_results) == 0:
    print("\n⚠️  WARNING: No forecast results available!")
    print("This likely means the forecasting cell didn't run or encountered errors.")
    print("Please run the forecasting cell first (the cell that calls run_sparkcaster_forecasting).")
    df_all_models = pd.DataFrame()  # Empty DataFrame
else:
    rows = []
    for result in all_results:
        if result.get('status') != 'completed':
            continue

        metric = result.get('metric')
        grouping = result.get('grouping') or result.get('grouping_name')
        group_key = result.get('group_key')
        best_model = result.get('best_model')

        for model_name, model_result in (result.get('results') or {}).items():
            if model_result.get('status') != 'success':
                continue

            metrics = model_result.get('metrics', {})
            rows.append({
                'metric': metric,
                'grouping': grouping,
                'group_key': group_key,
                'model': model_name,
                'is_best': model_name == best_model,
                'MSE': metrics.get('MSE'),
                'RMSE': metrics.get('RMSE'),
                'MAPE': metrics.get('MAPE'),
                'train_size': result.get('train_size'),
                'test_size': result.get('test_size'),
                'status': model_result.get('status')
            })

    df_all_models = pd.DataFrame(rows)

    print("\nAll Models Results:")
    print(f"Shape: {df_all_models.shape}")
    print(f"Columns: {list(df_all_models.columns)}")
    print("Sample data:")
    print(df_all_models.head(10))

# Save to Excel — via sandbox parquet round-trip, then read back and save locally
from datetime import datetime
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

# Export to CoreWeave Object Storage (CSV + XLSX)
save_df_to_s3('all_models_results', df_all_models)



All Models Results:
Shape: (1801, 11)
Columns: ['metric', 'grouping', 'group_key', 'model', 'is_best', 'MSE', 'RMSE', 'MAPE', 'train_size', 'test_size', 'status']
Sample data:
              metric                          grouping  group_key  \
0  redfish_power_p50                  product_resolved        L40   
1  redfish_power_p50                  product_resolved        L40   
2  redfish_power_p50                  product_resolved        L40   
3  redfish_power_p50                  product_resolved        L40   
4    tensor_util_p95  product_segment+product_resolved   pcie_L40   
5    tensor_util_p95  product_segment+product_resolved   pcie_L40   
6    tensor_util_p95  product_segment+product_resolved   pcie_L40   
7  redfish_power_p50   region_summary+product_resolved  NAM_GB200   
8  redfish_power_p50   region_summary+product_resolved  NAM_GB200   
9  redfish_power_p50   region_summary+product_resolved  NAM_GB200   

                   model  is_best           MSE         RMSE   

In [36]:
# 2. Create DataFrame for BEST models only
if df_all_models.empty:
    print("\n⚠️  WARNING: df_all_models is empty, cannot create best models DataFrame.")
    print("Please run the forecasting cell first.")
    df_best_models = pd.DataFrame()  # Empty DataFrame
elif 'is_best' not in df_all_models.columns:
    print("\n⚠️  WARNING: 'is_best' column not found in results.")
    print("This suggests the forecasting completed but didn't include best model selection.")
    df_best_models = pd.DataFrame()  # Empty DataFrame
else:
    df_best_models = df_all_models[df_all_models['is_best'] == True].copy()
    
    print("\nBest Models Results:")
    print(f"Shape: {df_best_models.shape}")
    print(f"\nSample data:")
    print(df_best_models.head(10))
    
    # Save to Excel
from datetime import datetime
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
# Export to CoreWeave Object Storage (CSV + XLSX)
save_df_to_s3('best_models_results', df_best_models)


Best Models Results:
Shape: (490, 11)

Sample data:
               metric                          grouping  group_key  \
0   redfish_power_p50                  product_resolved        L40   
6     tensor_util_p95  product_segment+product_resolved   pcie_L40   
9   redfish_power_p50   region_summary+product_resolved  NAM_GB200   
12    tensor_util_p95                  product_resolved     LEGACY   
15    tensor_util_p50  product_segment+product_resolved     LEGACY   
18    tensor_util_p99                  product_resolved        B40   
21    tensor_util_p50  product_segment+product_resolved   hgx_B200   
24    tensor_util_p50   region_summary+product_resolved   NAM_B200   
27       gpu_util_p50                  customer_segment     ai lab   
33  redfish_power_p95  product_segment+product_resolved  mgx_GB200   

                    model  is_best            MSE        RMSE          MAPE  \
0   exponential_smoothing     True   25566.784797  159.896169      6.346015   
6                 

In [37]:
from pathlib import Path
import os

print("cwd:", os.getcwd())
print("home:", str(Path.home()))

for p in [os.getcwd(), str(Path.home()), "/home/coreweave", "/tmp"]:
    print(p, "exists=", os.path.isdir(p), "writable=", os.access(p, os.W_OK))


cwd: /opt/spark/work-dir
home: /home/spark
/opt/spark/work-dir exists= True writable= True
/home/spark exists= True writable= True
/home/coreweave exists= False writable= False
/tmp exists= True writable= True


In [38]:
import os, glob
print("cwd:", os.getcwd())
print(glob.glob(os.path.join(os.getcwd(), "time_series_all_models_results_*.xlsx")))
print(glob.glob(os.path.join(str(Path.home()), "time_series_all_models_results_*.xlsx")))


cwd: /opt/spark/work-dir
[]
[]


In [39]:
# 3. Extract and save FULL DAILY FORECAST VALUES for best models
# This creates a detailed CSV with one row per day per time series

log_section("EXTRACTING FULL DAILY FORECAST VALUES")

forecast_details = []

for plot in plot_data_list:
    if plot is None:
        continue

    best_model = plot['best_model']
    best_result = plot['results'][best_model]

    # Get forecast array (point estimate)
    forecast_array = best_result.get('forecast')
    forecast_array = forecast_array.values if hasattr(forecast_array, 'values') else forecast_array

    # Get prediction intervals (lower and upper bounds)
    forecast_lower = best_result.get('forecast_lower')
    forecast_upper = best_result.get('forecast_upper')
    forecast_lower = forecast_lower.values if hasattr(forecast_lower, 'values') else forecast_lower
    forecast_upper = forecast_upper.values if hasattr(forecast_upper, 'values') else forecast_upper

    # Coerce to numpy arrays and align lengths
    forecast_array = np.asarray(forecast_array) if forecast_array is not None else np.array([])
    forecast_lower = np.asarray(forecast_lower) if forecast_lower is not None else np.array([])
    forecast_upper = np.asarray(forecast_upper) if forecast_upper is not None else np.array([])

    # If intervals are missing or length-mismatched, fall back to NaN arrays
    n = len(forecast_array)
    if len(forecast_lower) != n:
        forecast_lower = np.full(n, np.nan)
    if len(forecast_upper) != n:
        forecast_upper = np.full(n, np.nan)

    # Apply floor to prevent negative values (ignore NaNs)
    forecast_array = np.maximum(0, forecast_array)
    forecast_lower = np.where(np.isnan(forecast_lower), forecast_lower, np.maximum(0, forecast_lower))
    forecast_upper = np.where(np.isnan(forecast_upper), forecast_upper, np.maximum(0, forecast_upper))

    # Get metadata
    metadata = best_result['metadata']
    last_historical_date = metadata['train_dates'].max()

    # Create date range for forecast
    forecast_dates = pd.date_range(
        start=last_historical_date + pd.Timedelta(days=1),
        periods=n,
        freq='D'
    )

    # Create DataFrame for this time series forecast with prediction intervals
    df_forecast = pd.DataFrame({
        'metric': plot['metric'],
        'grouping': plot['grouping'],
        'group_key': plot['group_key'],
        'model': best_model,
        'forecast_date': forecast_dates,
        'forecast_value': forecast_array,  # Keep for backward compatibility
        'forecast_p50': forecast_array,    # Point estimate (median)
        'forecast_p10': forecast_lower,    # Lower confidence bound (10th percentile)
        'forecast_p90': forecast_upper,    # Upper confidence bound (90th percentile)
        'last_historical_date': last_historical_date,
        'forecast_horizon_days': range(1, n + 1)
    })

    forecast_details.append(df_forecast)

# Combine all forecasts
if forecast_details:
    df_all_forecasts = pd.concat(forecast_details, ignore_index=True)
else:
    df_all_forecasts = pd.DataFrame()

log("\nForecast Details:")
log(f"  Total rows: {len(df_all_forecasts):,}")
if not df_all_forecasts.empty:
    log(f"  Unique metrics: {df_all_forecasts['metric'].nunique()}")
    log(f"  Date range: {df_all_forecasts['forecast_date'].min()} to {df_all_forecasts['forecast_date'].max()}")

    log("\nSample data (with prediction intervals):")
    log(df_all_forecasts.head(10))

    # Save to CSV
    from datetime import datetime
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    # Export to CoreWeave Object Storage (CSV + XLSX)
    save_df_to_s3('forecast_daily_values', df_all_forecasts)
    log(f"\n✓ Saved {len(df_all_forecasts):,} daily forecast values")
    log(f"  Columns: {', '.join(df_all_forecasts.columns.tolist())}")
else:
    log("  No forecast details were generated.")

# Show utilization metrics specifically
log_section("UTILIZATION METRICS FORECAST RANGES")

if not df_all_forecasts.empty:
    util_forecasts = df_all_forecasts[df_all_forecasts['metric'].str.contains('util', case=False, na=False)]
    if len(util_forecasts) > 0:
        util_summary = util_forecasts.groupby(['metric', 'grouping', 'group_key']).agg({
            'forecast_p50': ['min', 'max', 'mean'],
            'forecast_p10': ['min', 'max', 'mean'],
            'forecast_p90': ['min', 'max', 'mean']
        }).round(4)

        log(f"Utilization forecast summary (P10/P50/P90):")
        log(util_summary.head(20))

        # Check bounds for all percentiles (ignore NaNs)
        max_p90 = util_forecasts['forecast_p90'].max()
        min_p10 = util_forecasts['forecast_p10'].min()
        max_p50 = util_forecasts['forecast_p50'].max()

        log_section("BOUNDS CHECK (with prediction intervals)")
        log(f"Global min forecast P10 value: {min_p10:.6f}")
        log(f"Global max forecast P50 value: {max_p50:.6f}")
        log(f"Global max forecast P90 value: {max_p90:.6f}")

        if max_p90 > 1.0:
            exceeds = util_forecasts[util_forecasts['forecast_p90'] > 1.0]
            log(f"⚠️  WARNING: {len(exceeds):,} P90 values exceed 1.0 (need to restart kernel and re-run)")
        else:
            log(f"✓ All utilization P90 forecasts within [0, 1] bounds")

        if min_p10 < 0.0:
            below = util_forecasts[util_forecasts['forecast_p10'] < 0.0]
            log(f"⚠️  WARNING: {len(below):,} P10 values below 0.0 (need to restart kernel and re-run)")
        else:
            log(f"✓ All utilization P10 forecasts >= 0.0")
    else:
        log("No utilization metrics found in forecast data")
else:
    log("No utilization metrics found in forecast data")

log("")
log_section(f"✓ Daily forecast extraction complete (with P10/P50/P90 prediction intervals)")


EXTRACTING FULL DAILY FORECAST VALUES

Forecast Details:
  Total rows: 539,000
  Unique metrics: 8
  Date range: 2026-01-03 00:00:00 to 2029-05-19 00:00:00

Sample data (with prediction intervals):
              metric          grouping group_key                  model  \
0  redfish_power_p50  product_resolved       L40  exponential_smoothing   
1  redfish_power_p50  product_resolved       L40  exponential_smoothing   
2  redfish_power_p50  product_resolved       L40  exponential_smoothing   
3  redfish_power_p50  product_resolved       L40  exponential_smoothing   
4  redfish_power_p50  product_resolved       L40  exponential_smoothing   
5  redfish_power_p50  product_resolved       L40  exponential_smoothing   
6  redfish_power_p50  product_resolved       L40  exponential_smoothing   
7  redfish_power_p50  product_resolved       L40  exponential_smoothing   
8  redfish_power_p50  product_resolved       L40  exponential_smoothing   
9  redfish_power_p50  product_resolved       L40  ex

In [40]:
# 3b. Extract DAILY FORECAST VALUES for ALL MODELS
# This creates a detailed DataFrame with one row per day per time series per model

log_section("EXTRACTING DAILY FORECAST VALUES FOR ALL MODELS")

forecast_details_all = []

for plot in plot_data_list:
    if plot is None:
        continue

    # Loop through ALL models, not just the best one
    for model_name, model_result in plot['results'].items():

        # Get forecast array (point estimate)
        forecast_array = model_result.get('forecast')
        forecast_array = forecast_array.values if hasattr(forecast_array, 'values') else forecast_array

        # Get prediction intervals (lower and upper bounds)
        forecast_lower = model_result.get('forecast_lower')
        forecast_upper = model_result.get('forecast_upper')
        forecast_lower = forecast_lower.values if hasattr(forecast_lower, 'values') else forecast_lower
        forecast_upper = forecast_upper.values if hasattr(forecast_upper, 'values') else forecast_upper

        # Coerce to numpy arrays and align lengths
        forecast_array = np.asarray(forecast_array) if forecast_array is not None else np.array([])
        forecast_lower = np.asarray(forecast_lower) if forecast_lower is not None else np.array([])
        forecast_upper = np.asarray(forecast_upper) if forecast_upper is not None else np.array([])

        n = len(forecast_array)
        if len(forecast_lower) != n:
            forecast_lower = np.full(n, np.nan)
        if len(forecast_upper) != n:
            forecast_upper = np.full(n, np.nan)

        # Apply floor to prevent negative values (ignore NaNs)
        forecast_array = np.maximum(0, forecast_array)
        forecast_lower = np.where(np.isnan(forecast_lower), forecast_lower, np.maximum(0, forecast_lower))
        forecast_upper = np.where(np.isnan(forecast_upper), forecast_upper, np.maximum(0, forecast_upper))

        # Get metadata
        metadata = model_result['metadata']
        last_historical_date = metadata['train_dates'].max()

        # Create date range for forecast
        forecast_dates = pd.date_range(
            start=last_historical_date + pd.Timedelta(days=1),
            periods=n,
            freq='D'
        )

        # Create DataFrame for this time series forecast with prediction intervals
        df_forecast = pd.DataFrame({
            'metric': plot['metric'],
            'grouping': plot['grouping'],
            'group_key': plot['group_key'],
            'model': model_name,
            'is_best_model': (model_name == plot['best_model']),
            'forecast_date': forecast_dates,
            'forecast_value': forecast_array,  # Keep for backward compatibility
            'forecast_p50': forecast_array,    # Point estimate (median)
            'forecast_p10': forecast_lower,    # Lower confidence bound (10th percentile)
            'forecast_p90': forecast_upper,    # Upper confidence bound (90th percentile)
            'last_historical_date': last_historical_date,
            'forecast_horizon_days': range(1, n + 1)
        })

        forecast_details_all.append(df_forecast)

# Combine all forecasts from all models
if forecast_details_all:
    df_all_models_forecasts = pd.concat(forecast_details_all, ignore_index=True)
else:
    df_all_models_forecasts = pd.DataFrame()

log(f"All Models Forecast Details:")
log(f"  Total rows: {len(df_all_models_forecasts):,}")
if not df_all_models_forecasts.empty:
    log(f"  Unique time series: {len(df_all_models_forecasts.groupby(['metric', 'grouping', 'group_key']))}")
    log(f"  Unique models: {df_all_models_forecasts['model'].nunique()}")
    log(f"  Models: {sorted(df_all_models_forecasts['model'].unique())}")
    log(f"  Unique metrics: {df_all_models_forecasts['metric'].nunique()}")
    log(f"  Date range: {df_all_models_forecasts['forecast_date'].min()} to {df_all_models_forecasts['forecast_date'].max()}")

    log(f"Model breakdown:")
    log(df_all_models_forecasts.groupby('model').size())

    log(f"Sample data (with all models):")
    log(df_all_models_forecasts.head(15))

log_section(f"✓ All models daily forecast extraction complete")


EXTRACTING DAILY FORECAST VALUES FOR ALL MODELS
All Models Forecast Details:
  Total rows: 1,981,100
  Unique time series: 490
  Unique models: 4
  Models: ['arima', 'exponential_smoothing', 'holt_winters', 'sarima']
  Unique metrics: 8
  Date range: 2026-01-03 00:00:00 to 2029-05-19 00:00:00
Model breakdown:
model
arima                    539000
exponential_smoothing    539000
holt_winters             364100
sarima                   539000
dtype: int64
Sample data (with all models):
               metric          grouping group_key                  model  \
0   redfish_power_p50  product_resolved       L40  exponential_smoothing   
1   redfish_power_p50  product_resolved       L40  exponential_smoothing   
2   redfish_power_p50  product_resolved       L40  exponential_smoothing   
3   redfish_power_p50  product_resolved       L40  exponential_smoothing   
4   redfish_power_p50  product_resolved       L40  exponential_smoothing   
5   redfish_power_p50  product_resolved       L40  expo

In [41]:
# 3c. AGGREGATE DAILY FORECASTS TO MONTHLY GRAIN - ALL MODELS
# This creates a CSV with monthly aggregated forecast values for ALL models

log("")
log_section("AGGREGATING DAILY FORECASTS TO MONTHLY GRAIN - ALL MODELS")

# Add year-month column for grouping
df_all_models_forecasts['year_month'] = df_all_models_forecasts['forecast_date'].dt.to_period('M')

# Group by time series identifiers, MODEL, and year-month, then aggregate
monthly_forecasts_all = df_all_models_forecasts.groupby([
    'metric', 
    'grouping', 
    'group_key', 
    'model',
    'year_month'
]).agg({
    'is_best_model': 'first',  # Boolean flag - same for all days in month
    'forecast_value': 'mean',  # Average daily values for the month (backward compatibility)
    'forecast_p50': 'mean',    # Average P50 point estimates for the month
    'forecast_p10': 'mean',    # Average P10 lower bounds for the month
    'forecast_p90': 'mean',    # Average P90 upper bounds for the month
    'forecast_date': ['min', 'max'],  # First and last date in month
    'last_historical_date': 'first',
    'forecast_horizon_days': ['min', 'max']  # Min and max horizon days in month
}).reset_index()

# Flatten column names
monthly_forecasts_all.columns = [
    'metric', 'grouping', 'group_key', 'model', 'year_month',
    'is_best_model',
    'avg_forecast_value',
    'avg_forecast_p50', 
    'avg_forecast_p10', 
    'avg_forecast_p90',
    'month_start_date', 'month_end_date',
    'last_historical_date',
    'forecast_horizon_days_min', 'forecast_horizon_days_max'
]

# Convert year_month back to string for better CSV readability
monthly_forecasts_all['year_month'] = monthly_forecasts_all['year_month'].astype(str)

# Add a column for number of days in the forecast month period
monthly_forecasts_all['days_in_month_period'] = (
    pd.to_datetime(monthly_forecasts_all['month_end_date']) - 
    pd.to_datetime(monthly_forecasts_all['month_start_date'])
).dt.days + 1

log(f"\nMonthly Forecast Summary (All Models):")
log(f"  Total rows: {len(monthly_forecasts_all):,}")
log(f"  Unique time series: {len(monthly_forecasts_all.groupby(['metric', 'grouping', 'group_key']))}")
log(f"  Unique models: {monthly_forecasts_all['model'].nunique()}")
log(f"  Models: {sorted(monthly_forecasts_all['model'].unique())}")
log(f"  Unique metrics: {monthly_forecasts_all['metric'].nunique()}")
log(f"  Date range: {monthly_forecasts_all['year_month'].min()} to {monthly_forecasts_all['year_month'].max()}")

log(f"\nRows per model:")
log(monthly_forecasts_all.groupby('model').size())

log(f"\nBest model flags:")
log(monthly_forecasts_all.groupby('is_best_model').size())

log(f"\nSample monthly data (all models):")
log(monthly_forecasts_all.head(20))

# Save to CSV
from datetime import datetime
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
# Export to CoreWeave Object Storage (CSV + XLSX)
save_df_to_s3('forecast_monthly_values_all', monthly_forecasts_all)
log(f"\n✓ Saved {len(monthly_forecasts_all):,} monthly forecast values (all models)")

# Show comparison between best model only vs all models (if available)
try:
    log_section("COMPARISON: BEST MODEL ONLY vs ALL MODELS")
    log(f"Best model only CSV rows: {len(monthly_forecasts):,}")
    log(f"All models CSV rows: {len(monthly_forecasts_all):,}")
    log(f"Ratio: {len(monthly_forecasts_all) / len(monthly_forecasts):.2f}x more rows")
except NameError:
    log_section("ALL MODELS CSV CREATED")
    log(f"All models CSV rows: {len(monthly_forecasts_all):,}")
    log("(Run cell 36 to create best-model-only CSV for comparison)")

# Show utilization metrics at monthly grain for all models
log("")
log_section("UTILIZATION METRICS - MONTHLY AGGREGATION (ALL MODELS)")

util_monthly_all = monthly_forecasts_all[monthly_forecasts_all['metric'].str.contains('util', case=False)]
if len(util_monthly_all) > 0:
    log(f"\nMonthly utilization forecasts (all models):")
    log(f"  Rows: {len(util_monthly_all):,}")
    log(f"  Models: {sorted(util_monthly_all['model'].unique())}")
    
    # Check bounds
    max_p50 = util_monthly_all['avg_forecast_p50'].max()
    min_p50 = util_monthly_all['avg_forecast_p50'].min()
    max_p10 = util_monthly_all['avg_forecast_p10'].max()
    min_p10 = util_monthly_all['avg_forecast_p10'].min()
    max_p90 = util_monthly_all['avg_forecast_p90'].max()
    min_p90 = util_monthly_all['avg_forecast_p90'].min()
    
    log("")
    log_section("MONTHLY BOUNDS CHECK (ALL MODELS)")
    log(f"P50 (point estimate) range: [{min_p50:.6f}, {max_p50:.6f}]")
    log(f"P10 (lower bound) range:    [{min_p10:.6f}, {max_p10:.6f}]")
    log(f"P90 (upper bound) range:    [{min_p90:.6f}, {max_p90:.6f}]")
    
    # Check P50/P10/P90 bounds
    if max_p50 > 1.0:
        exceeds = util_monthly_all[util_monthly_all['avg_forecast_p50'] > 1.0]
        log(f"⚠️  WARNING: {len(exceeds):,} monthly P50 values exceed 1.0")
        log(f"   Models with issues: {sorted(exceeds['model'].unique())}")
    else:
        log(f"✓ All monthly P50 utilization forecasts within [0, 1] bounds")
    
    if min_p50 < 0.0:
        below = util_monthly_all[util_monthly_all['avg_forecast_p50'] < 0.0]
        log(f"⚠️  WARNING: {len(below):,} monthly P50 values below 0.0")
        log(f"   Models with issues: {sorted(below['model'].unique())}")
    else:
        log(f"✓ All monthly P50 utilization forecasts >= 0.0")
    
    if max_p90 > 1.0:
        exceeds = util_monthly_all[util_monthly_all['avg_forecast_p90'] > 1.0]
        log(f"⚠️  WARNING: {len(exceeds):,} monthly P90 values exceed 1.0")
        log(f"   Models with issues: {sorted(exceeds['model'].unique())}")
    else:
        log(f"✓ All monthly P90 utilization forecasts within [0, 1] bounds")
    
    if min_p10 < 0.0:
        below = util_monthly_all[util_monthly_all['avg_forecast_p10'] < 0.0]
        log(f"⚠️  WARNING: {len(below):,} monthly P10 values below 0.0")
        log(f"   Models with issues: {sorted(below['model'].unique())}")
    else:
        log(f"✓ All monthly P10 utilization forecasts >= 0.0")
else:
    log("No utilization metrics found in monthly forecast data")

log(f"\n{'='*80}")
log(f"✓ Monthly forecast aggregation complete (ALL MODELS)")
log(f"  Output: s3://{S3_BUCKET}/{S3_PREFIX}forecast_monthly_values_all.csv")
log(f"{'='*80}")


AGGREGATING DAILY FORECASTS TO MONTHLY GRAIN - ALL MODELS

Monthly Forecast Summary (All Models):
  Total rows: 66,690
  Unique time series: 490
  Unique models: 4
  Models: ['arima', 'exponential_smoothing', 'holt_winters', 'sarima']
  Unique metrics: 8
  Date range: 2026-01 to 2029-05

Rows per model:
model
arima                    18145
exponential_smoothing    18145
holt_winters             12255
sarima                   18145
dtype: int64

Best model flags:
is_best_model
False    48545
True     18145
dtype: int64

Sample monthly data (all models):
            metric grouping group_key  model year_month  is_best_model  \
0   chip_power_p50      All       All  arima    2026-01           True   
1   chip_power_p50      All       All  arima    2026-02           True   
2   chip_power_p50      All       All  arima    2026-03           True   
3   chip_power_p50      All       All  arima    2026-04           True   
4   chip_power_p50      All       All  arima    2026-05           True

In [42]:
# 3d. ADD HISTORICAL DATA TO MONTHLY CSV (ALL MODELS)
# This extends the monthly CSV to include actual historical values before forecasts

log("")
log_section("ADDING HISTORICAL DATA TO MONTHLY FORECASTS - ALL MODELS")

historical_monthly = []

# Extract historical data from plot_data_list
for plot in plot_data_list:
    if plot is None:
        continue
    
    # Get historical data from any model's metadata (same for all models)
    first_model = list(plot['results'].keys())[0]
    metadata = plot['results'][first_model]['metadata']
    
    train_dates = pd.to_datetime(metadata['train_dates'])
    test_dates = pd.to_datetime(metadata.get('test_dates', []))
    train_values = pd.Series(metadata['train_values'])
    test_values = pd.Series(metadata.get('test_values', []))

    # Combine train + test to get full historical actuals
    full_dates = pd.concat([pd.Series(train_dates), pd.Series(test_dates)], ignore_index=True)
    full_values = pd.concat([train_values, test_values], ignore_index=True)

    # Create DataFrame with historical data
    df_hist = pd.DataFrame({
        'date': full_dates,
        'value': full_values,
        'metric': plot['metric'],
        'grouping': plot['grouping'],
        'group_key': plot['group_key']
    })
    
    # Add year-month column
    df_hist['year_month'] = pd.to_datetime(df_hist['date']).dt.to_period('M')
    
    # Group by month and calculate monthly averages
    monthly_hist = df_hist.groupby([
        'metric', 'grouping', 'group_key', 'year_month'
    ]).agg({
        'value': 'mean',
        'date': ['min', 'max']
    }).reset_index()
    
    # Flatten columns
    monthly_hist.columns = [
        'metric', 'grouping', 'group_key', 'year_month',
        'avg_actual_value', 'month_start_date', 'month_end_date'
    ]
    
    historical_monthly.append(monthly_hist)

# Combine all historical data
df_historical_monthly = pd.concat(historical_monthly, ignore_index=True)
df_historical_monthly['year_month'] = df_historical_monthly['year_month'].astype(str)

log(f"\nHistorical Monthly Data:")
log(f"  Total rows: {len(df_historical_monthly):,}")
log(f"  Unique time series: {len(df_historical_monthly.groupby(['metric', 'grouping', 'group_key']))}")
log(f"  Date range: {df_historical_monthly['year_month'].min()} to {df_historical_monthly['year_month'].max()}")

# Now merge with forecast data
# For historical data, we need to add model column and replicate for each model
models_list = monthly_forecasts_all['model'].unique()
log(f"\nReplicating historical data for {len(models_list)} models: {sorted(models_list)}")

# Replicate historical data for each model
historical_replicated = []
for model in models_list:
    df_hist_model = df_historical_monthly.copy()
    df_hist_model['model'] = model
    historical_replicated.append(df_hist_model)

df_historical_all_models = pd.concat(historical_replicated, ignore_index=True)

# Add columns to match forecast structure
df_historical_all_models['is_best_model'] = False  # Not applicable for historical
df_historical_all_models['avg_forecast_value'] = None
df_historical_all_models['avg_forecast_p50'] = None
df_historical_all_models['avg_forecast_p10'] = None
df_historical_all_models['avg_forecast_p90'] = None
df_historical_all_models['last_historical_date'] = None
df_historical_all_models['forecast_horizon_days_min'] = None
df_historical_all_models['forecast_horizon_days_max'] = None
df_historical_all_models['days_in_month_period'] = (
    pd.to_datetime(df_historical_all_models['month_end_date']) - 
    pd.to_datetime(df_historical_all_models['month_start_date'])
).dt.days + 1
df_historical_all_models['data_type'] = 'actual'

# Add data_type to forecast data
monthly_forecasts_all['avg_actual_value'] = None
monthly_forecasts_all['data_type'] = 'forecast'

# Ensure column order matches
common_columns = [
    'metric', 'grouping', 'group_key', 'model', 'year_month',
    'is_best_model', 'data_type',
    'avg_actual_value', 'avg_forecast_value',
    'avg_forecast_p50', 'avg_forecast_p10', 'avg_forecast_p90',
    'month_start_date', 'month_end_date',
    'last_historical_date',
    'forecast_horizon_days_min', 'forecast_horizon_days_max',
    'days_in_month_period'
]

# Reorder columns
df_historical_all_models = df_historical_all_models[common_columns]
monthly_forecasts_all_ordered = monthly_forecasts_all[common_columns]

# Combine historical + forecast
monthly_forecasts_with_history = pd.concat([
    df_historical_all_models,
    monthly_forecasts_all_ordered
], ignore_index=True)

# Sort by time series, model, and date
monthly_forecasts_with_history = monthly_forecasts_with_history.sort_values([
    'metric', 'grouping', 'group_key', 'model', 'year_month'
]).reset_index(drop=True)

log(f"\nCombined Historical + Forecast Data:")
log(f"  Total rows: {len(monthly_forecasts_with_history):,}")
log(f"  Historical rows: {len(monthly_forecasts_with_history[monthly_forecasts_with_history['data_type']=='actual']):,}")
log(f"  Forecast rows: {len(monthly_forecasts_with_history[monthly_forecasts_with_history['data_type']=='forecast']):,}")
log(f"  Date range: {monthly_forecasts_with_history['year_month'].min()} to {monthly_forecasts_with_history['year_month'].max()}")

log(f"\nSample data (showing transition from actual to forecast):")
# Show sample for one time series
sample_ts = monthly_forecasts_with_history.head(50)
log(sample_ts[['metric', 'grouping', 'group_key', 'model', 'year_month', 'data_type', 'avg_actual_value', 'avg_forecast_p50']].head(20))

# Save combined file
from datetime import datetime
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
# Export to CoreWeave Object Storage (CSV + XLSX)
save_df_to_s3('forecast_monthly_values_all_with_history', monthly_forecasts_with_history)
log(f"\n✓ Saved {len(monthly_forecasts_with_history):,} monthly values (historical + forecast, all models)")

log(f"\n{'='*80}")
log(f"✓ Historical + Forecast monthly data creation complete")
log(f"  - Use 'data_type' column to filter: 'actual' vs 'forecast'")
log(f"  - Historical data: avg_actual_value column")
log(f"  - Forecast data: avg_forecast_p50, avg_forecast_p10, avg_forecast_p90 columns")
log(f"{'='*80}")



ADDING HISTORICAL DATA TO MONTHLY FORECASTS - ALL MODELS

Historical Monthly Data:
  Total rows: 6,997
  Unique time series: 490
  Date range: 2025-01 to 2026-05

Replicating historical data for 4 models: ['arima', 'exponential_smoothing', 'holt_winters', 'sarima']

Combined Historical + Forecast Data:
  Total rows: 94,678
  Historical rows: 27,988
  Forecast rows: 66,690
  Date range: 2025-01 to 2029-05

Sample data (showing transition from actual to forecast):
            metric grouping group_key  model year_month data_type  \
0   chip_power_p50      All       All  arima    2025-01    actual   
1   chip_power_p50      All       All  arima    2025-02    actual   
2   chip_power_p50      All       All  arima    2025-03    actual   
3   chip_power_p50      All       All  arima    2025-04    actual   
4   chip_power_p50      All       All  arima    2025-05    actual   
5   chip_power_p50      All       All  arima    2025-06    actual   
6   chip_power_p50      All       All  arima    2

In [43]:
# 4. AGGREGATE DAILY FORECASTS TO MONTHLY GRAIN
# This creates a CSV with monthly aggregated forecast values

log("")
log_section("AGGREGATING DAILY FORECASTS TO MONTHLY GRAIN")

# Add year-month column for grouping
df_all_forecasts['year_month'] = df_all_forecasts['forecast_date'].dt.to_period('M')

# Group by time series identifiers and year-month, then aggregate
monthly_forecasts = df_all_forecasts.groupby([
    'metric', 
    'grouping', 
    'group_key', 
    'model', 
    'year_month'
]).agg({
    'forecast_value': 'mean',  # Average daily values for the month (backward compatibility)
    'forecast_p50': 'mean',    # Average P50 point estimates for the month
    'forecast_p10': 'mean',    # Average P10 lower bounds for the month
    'forecast_p90': 'mean',    # Average P90 upper bounds for the month
    'forecast_date': ['min', 'max'],  # First and last date in month
    'last_historical_date': 'first',
    'forecast_horizon_days': ['min', 'max']  # Min and max horizon days in month
}).reset_index()

# Flatten column names
monthly_forecasts.columns = [
    'metric', 'grouping', 'group_key', 'model', 'year_month',
    'avg_forecast_value',
    'avg_forecast_p50', 
    'avg_forecast_p10', 
    'avg_forecast_p90',
    'month_start_date', 'month_end_date',
    'last_historical_date',
    'forecast_horizon_days_min', 'forecast_horizon_days_max'
]

# Convert year_month back to string for better CSV readability
monthly_forecasts['year_month'] = monthly_forecasts['year_month'].astype(str)

# Add a column for number of days in the forecast month period
monthly_forecasts['days_in_month_period'] = (
    pd.to_datetime(monthly_forecasts['month_end_date']) - 
    pd.to_datetime(monthly_forecasts['month_start_date'])
).dt.days + 1

log(f"\nMonthly Forecast Summary:")
log(f"  Total rows: {len(monthly_forecasts):,}")
log(f"  Unique time series: {len(monthly_forecasts.groupby(['metric', 'grouping', 'group_key']))}")
log(f"  Unique metrics: {monthly_forecasts['metric'].nunique()}")
log(f"  Date range: {monthly_forecasts['year_month'].min()} to {monthly_forecasts['year_month'].max()}")

log(f"\nSample monthly data:")
log(monthly_forecasts.head(10))

# Save to CSV
from datetime import datetime
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
# Export to CoreWeave Object Storage (CSV + XLSX)
save_df_to_s3('forecast_monthly_values', monthly_forecasts)
log(f"\n✓ Saved {len(monthly_forecasts):,} monthly forecast values")

# Show utilization metrics at monthly grain
log("")
log_section("UTILIZATION METRICS - MONTHLY AGGREGATION")

util_monthly = monthly_forecasts[monthly_forecasts['metric'].str.contains('util', case=False)]
if len(util_monthly) > 0:
    log(f"\nMonthly utilization forecasts:")
    log(f"  Rows: {len(util_monthly):,}")
    
    util_monthly_summary = util_monthly.groupby(['metric', 'grouping', 'group_key']).agg({
        'avg_forecast_p50': ['min', 'max', 'mean'],
        'avg_forecast_p10': ['min', 'max', 'mean'],
        'avg_forecast_p90': ['min', 'max', 'mean']
    }).round(4)
    
    log(f"\nUtilization monthly summary (by time series):")
    log(util_monthly_summary.head(20))
    
    # Bounds check for P50, P10, and P90
    max_p50 = util_monthly['avg_forecast_p50'].max()
    min_p50 = util_monthly['avg_forecast_p50'].min()
    max_p10 = util_monthly['avg_forecast_p10'].max()
    min_p10 = util_monthly['avg_forecast_p10'].min()
    max_p90 = util_monthly['avg_forecast_p90'].max()
    min_p90 = util_monthly['avg_forecast_p90'].min()
    
    log("")
    log_section("MONTHLY BOUNDS CHECK")
    log(f"P50 (point estimate) range: [{min_p50:.6f}, {max_p50:.6f}]")
    log(f"P10 (lower bound) range:    [{min_p10:.6f}, {max_p10:.6f}]")
    log(f"P90 (upper bound) range:    [{min_p90:.6f}, {max_p90:.6f}]")
    
    # Check P50 bounds
    if max_p50 > 1.0:
        exceeds = util_monthly[util_monthly['avg_forecast_p50'] > 1.0]
        log(f"⚠️  WARNING: {len(exceeds):,} monthly P50 values exceed 1.0")
    else:
        log(f"✓ All monthly P50 utilization forecasts within [0, 1] bounds")
    
    if min_p50 < 0.0:
        below = util_monthly[util_monthly['avg_forecast_p50'] < 0.0]
        log(f"⚠️  WARNING: {len(below):,} monthly P50 values below 0.0")
    else:
        log(f"✓ All monthly P50 utilization forecasts >= 0.0")
    
    # Check P10 bounds
    if max_p10 > 1.0:
        exceeds = util_monthly[util_monthly['avg_forecast_p10'] > 1.0]
        log(f"⚠️  WARNING: {len(exceeds):,} monthly P10 values exceed 1.0")
    else:
        log(f"✓ All monthly P10 utilization forecasts within [0, 1] bounds")
    
    if min_p10 < 0.0:
        below = util_monthly[util_monthly['avg_forecast_p10'] < 0.0]
        log(f"⚠️  WARNING: {len(below):,} monthly P10 values below 0.0")
    else:
        log(f"✓ All monthly P10 utilization forecasts >= 0.0")
    
    # Check P90 bounds
    if max_p90 > 1.0:
        exceeds = util_monthly[util_monthly['avg_forecast_p90'] > 1.0]
        log(f"⚠️  WARNING: {len(exceeds):,} monthly P90 values exceed 1.0")
    else:
        log(f"✓ All monthly P90 utilization forecasts within [0, 1] bounds")
    
    if min_p90 < 0.0:
        below = util_monthly[util_monthly['avg_forecast_p90'] < 0.0]
        log(f"⚠️  WARNING: {len(below):,} monthly P90 values below 0.0")
    else:
        log(f"✓ All monthly P90 utilization forecasts >= 0.0")
else:
    log("No utilization metrics found in monthly forecast data")

log(f"\n{'='*80}")
log(f"✓ Monthly forecast aggregation complete")
log(f"  Output: s3://{S3_BUCKET}/{S3_PREFIX}forecast_monthly_values.csv")
log(f"{'='*80}")



AGGREGATING DAILY FORECASTS TO MONTHLY GRAIN

Monthly Forecast Summary:
  Total rows: 18,145
  Unique time series: 490
  Unique metrics: 8
  Date range: 2026-01 to 2029-05

Sample monthly data:
           metric grouping group_key  model year_month  avg_forecast_value  \
0  chip_power_p50      All       All  arima    2026-01         1292.655289   
1  chip_power_p50      All       All  arima    2026-02         1292.655289   
2  chip_power_p50      All       All  arima    2026-03         1292.655289   
3  chip_power_p50      All       All  arima    2026-04         1292.655289   
4  chip_power_p50      All       All  arima    2026-05         1292.655289   
5  chip_power_p50      All       All  arima    2026-06         1292.655289   
6  chip_power_p50      All       All  arima    2026-07         1292.655289   
7  chip_power_p50      All       All  arima    2026-08         1292.655289   
8  chip_power_p50      All       All  arima    2026-09         1292.655289   
9  chip_power_p50      Al

In [44]:
# Summary Statistics
log("")
log_section("SUMMARY STATISTICS")

log(f"\nTotal model runs: {len(df_all_models)}")
log(f"Total best models selected: {len(df_best_models)}")
log(f"Total plots generated: {len(all_plots)}")

log("\n--- Best Model Distribution ---")
log(df_best_models['model'].value_counts())

log("\n--- Average Error Metrics by Model (Best Models Only) ---")
log(df_best_models.groupby('model')[['MSE', 'RMSE', 'MAPE']].mean())

log("\n--- Best Models by Metric ---")
for metric in METRICS:
    metric_best = df_best_models[df_best_models['metric'] == metric]
    if len(metric_best) > 0:
        log(f"\n{metric}:")
        log(f"  Most common best model: {metric_best['model'].mode().values[0] if len(metric_best['model'].mode()) > 0 else 'N/A'}")
        log(f"  Avg RMSE: {metric_best['RMSE'].mean():.4f}")
        log(f"  Avg MAPE: {metric_best['MAPE'].mean():.2f}%")

log("")
log_section("FILES GENERATED (CoreWeave Object Storage)")
print_s3_manifest()
log("="*80)


SUMMARY STATISTICS

Total model runs: 1801
Total best models selected: 490
Total plots generated: 490

--- Best Model Distribution ---
model
arima                    271
exponential_smoothing     84
sarima                    75
holt_winters              60
Name: count, dtype: int64

--- Average Error Metrics by Model (Best Models Only) ---
                                 MSE        RMSE         MAPE
model                                                        
arima                  196128.014031  215.514001  1089.581136
exponential_smoothing  187823.631304  249.775045    65.399965
holt_winters           159567.010169  252.561421   331.214085
sarima                 442817.400931  402.225435    42.634994

--- Best Models by Metric ---

gpu_util_p50:
  Most common best model: arima
  Avg RMSE: 0.1185
  Avg MAPE: 181.47%

tensor_util_p50:
  Most common best model: arima
  Avg RMSE: 0.0220
  Avg MAPE: 2519.96%

tensor_util_p95:
  Most common best model: arima
  Avg RMSE: 0.0834
  Avg MAP

In [45]:
# PERFORMANCE ANALYSIS
log("")
log_section("PERFORMANCE ANALYSIS")

# Calculate performance metrics
total_model_runs = len(df_all_models)
total_time_series = len(df_best_models)
avg_time_per_series = forecast_time / total_time_series if total_time_series > 0 else 0

log(f"\nThroughput Metrics:")
log(f"  Total model runs: {total_model_runs}")
log(f"  Unique time series: {total_time_series}")
log(f"  Total time: {forecast_time:.2f}s ({forecast_time/60:.2f}m)")
log(f"  Time per series: {avg_time_per_series:.2f}s")
log(f"  Series per second: {total_time_series/forecast_time:.2f}")

# Estimate sequential time
sequential_time = forecast_time * CONFIG['n_workers']
speedup = sequential_time / forecast_time if forecast_time > 0 else 0

log(f"\nParallel Efficiency:")
log(f"  Workers used: {CONFIG['n_workers']}")
log(f"  Estimated sequential time: {sequential_time/60:.1f} minutes")
log(f"  Actual parallel time: {forecast_time/60:.1f} minutes")
log(f"  Speedup: {speedup:.1f}x")
log(f"  Parallel efficiency: {(speedup/CONFIG['n_workers']*100):.1f}%")

log(f"\nResource Utilization:")
log(f"  CPU cores: 32 available, {CONFIG['n_workers']} used ({CONFIG['n_workers']/32*100:.0f}%)")
log(f"  RAM: 512GB available")
log(f"  Spark cluster: Available but not used (multiprocessing sufficient for this dataset)")

log("")
log_section("OPTIMIZATION RECOMMENDATIONS")

if total_time_series < 100:
    log("\n✓ Dataset size: SMALL (< 100 time series)")
    log("  Recommendation: Current parallel processing is optimal")
    log("  Alternative: Could use sequential processing if needed")
    
elif total_time_series < 500:
    log("\n✓ Dataset size: MEDIUM (100-500 time series)")
    log("  Recommendation: Standard parallel processing (current method) is optimal")
    log("  Workers: 30 is good, could increase to 31 for marginal gains")
    
elif total_time_series < 2000:
    log("\n⚡ Dataset size: LARGE (500-2000 time series)")
    log("  Recommendation: Consider chunked processing for better memory management")
    log("  Set: USE_CHUNKED = True, chunk_size = 150")
    
else:
    log("\n⚡⚡ Dataset size: VERY LARGE (> 2000 time series)")
    log("  Recommendation: Use chunked processing")
    log("  Set: USE_CHUNKED = True, chunk_size = 100-200")
    log("  Consider: Spark distributed processing for > 5000 time series")

log(f"\n{'='*80}")


PERFORMANCE ANALYSIS

Throughput Metrics:
  Total model runs: 1801
  Unique time series: 490
  Total time: 37.80s (0.63m)
  Time per series: 0.08s
  Series per second: 12.96

Parallel Efficiency:
  Workers used: 30
  Estimated sequential time: 18.9 minutes
  Actual parallel time: 0.6 minutes
  Speedup: 30.0x
  Parallel efficiency: 100.0%

Resource Utilization:
  CPU cores: 32 available, 30 used (94%)
  RAM: 512GB available
  Spark cluster: Available but not used (multiprocessing sufficient for this dataset)

OPTIMIZATION RECOMMENDATIONS

✓ Dataset size: MEDIUM (100-500 time series)
  Recommendation: Standard parallel processing (current method) is optimal
  Workers: 30 is good, could increase to 31 for marginal gains



In [46]:
# For Kubeflow, you need to use the file browser
print("To download from Kubeflow Notebooks:")
print("=" * 80)
print("\n1. Look at the LEFT SIDEBAR in JupyterLab")
print("2. Click the FOLDER icon (File Browser)")
print("3. Find 'forecasts.zip' in the file list")
print("4. RIGHT-CLICK on 'forecasts.zip'")
print("5. Select 'Download'")
print("\nAlternatively:")
print("- Go to the URL: /files/forecasts.zip")
print("- Or click on the file and use the Download button in the toolbar")
print("\n" + "=" * 80)
print("\nFile ready to download:")
print("  forecasts.zip (20 MB - contains time_series_forecasts.html)")

To download from Kubeflow Notebooks:

1. Look at the LEFT SIDEBAR in JupyterLab
2. Click the FOLDER icon (File Browser)
3. Find 'forecasts.zip' in the file list
4. RIGHT-CLICK on 'forecasts.zip'
5. Select 'Download'

Alternatively:
- Go to the URL: /files/forecasts.zip
- Or click on the file and use the Download button in the toolbar


File ready to download:
  forecasts.zip (20 MB - contains time_series_forecasts.html)
